# Lab 9 - Data Cleaning and Pipeline Integration
**Domain Context:** Art Museum Collection Data Processing Pipeline  

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
SRC_DIR = PROJECT_ROOT / "src"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from src.analytics.data_loader import chunked_stats

from src.cleaning.type_converter import  memory_report, optimise_dtypes

In [2]:
print("--- Step 1: Loading Raw Data ---")
raw_data_path = PROJECT_ROOT / "data" / "processed" / "analytics" / "raw_art_data.csv"
df_raw = pd.read_csv(raw_data_path)
print(f"Loaded Raw Shape: {df_raw.shape}")

print("\n--- Step 2: Generating Missing Data Report ---")
missing_counts = df_raw.isnull().sum()
missing_pct = (df_raw.isnull().sum() / len(df_raw)) * 100
missing_report = pd.DataFrame({"total_missing": missing_counts, "missing_ratio_pct": missing_pct})

output_report_path = PROJECT_ROOT / "data" / "processed" / "cleaned" / "missing_report.csv"
output_report_path.parent.mkdir(parents=True, exist_ok=True)
missing_report.to_csv(output_report_path)
print(f"Saved missing report summary to: {output_report_path}")
missing_report

--- Step 1: Loading Raw Data ---
Loaded Raw Shape: (5, 11)

--- Step 2: Generating Missing Data Report ---
Saved missing report summary to: c:\xampp\htdocs\Art-Museum-Collection-Pipeline\data\processed\cleaned\missing_report.csv


,total_missing,missing_ratio_pct
id,0,0.0
title,0,0.0
artist_display,0,0.0
date_start,0,0.0
date_display,0,0.0
department,0,0.0
classification,0,0.0
objectName,0,0.0
language,1,20.0
description,0,0.0


In [3]:
print("--- Step 3: String Clean Preview ---")
df_strings = df_raw.copy()

for text_col in ["title", "artist_display", "department", "classification"]:
    if text_col in df_strings.columns:
        df_strings[text_col] = df_strings[text_col].astype(str).str.strip().str.title()

if "language" in df_strings.columns:
    df_strings["language"] = df_strings["language"].astype(str).str.strip().str.lower()

if "date_display" in df_strings.columns:
    df_strings["release_year"] = df_strings["date_display"].str.extract(r'(\d{4})')
    df_strings["release_year"] = pd.to_numeric(df_strings["release_year"], errors="coerce").astype("Int64")

df_strings[["title", "artist_display", "language", "release_year"]].head()

--- Step 3: String Clean Preview ---


,title,artist_display,language,release_year
0,Starry Night,Vincent Van Gogh,en,1889
1,The Scream,Edvard Munch,no,1893
2,Mona Lisa,Leonardo Da Vinci,NaN,1503
3,The Kiss,Gustav Klimt,de,1907
4,The Night Watch,Rembrandt,nl,1642


In [4]:
print("--- Step 4: Duplicate Analysis ---")
print(f"Total rows before deduplication: {len(df_strings)}")

if "id" in df_strings.columns:
    duplicate_id_count = df_strings.duplicated(subset=["id"]).sum()
    print(f"Key Duplicates detected on 'id' attribute: {duplicate_id_count}")

df_dedup = df_strings.drop_duplicates().copy()
if "id" in df_dedup.columns:
    df_dedup = df_dedup.drop_duplicates(subset=["id"], keep="first").copy()

print(f"Total rows remaining after sequence completion: {len(df_dedup)}")

--- Step 4: Duplicate Analysis ---
Total rows before deduplication: 5
Key Duplicates detected on 'id' attribute: 0
Total rows remaining after sequence completion: 5


In [5]:
print("--- Step 5: High-Performance Datatype Conversions ---")
print(f"Initial Memory Footprint: {df_dedup.memory_usage(deep=True).sum()} bytes")

df_converted = df_dedup.copy()

if "date_start" in df_converted.columns:
    df_converted["date_start"] = pd.to_numeric(df_converted["date_start"], errors="coerce").astype("Int64")

for cat_col in ["department", "classification", "language"]:
    if cat_col in df_converted.columns:
        df_converted[cat_col] = df_converted[cat_col].astype("category")

print(f"Optimized Memory Footprint: {df_converted.memory_usage(deep=True).sum()} bytes")
print("\nFinal Formatted Datatype Layout:")
print(df_converted.dtypes)

--- Step 5: High-Performance Datatype Conversions ---
Initial Memory Footprint: 2635 bytes
Optimized Memory Footprint: 2395 bytes

Final Formatted Datatype Layout:
id                   int64
title                  str
artist_display         str
date_start           Int64
date_display           str
department        category
classification    category
objectName             str
language          category
description            str
release_year         Int64
dtype: object


In [6]:
print("--- Step 6: Final Structural System Validation ---")

assert df_converted["title"].notnull().all(), "Validation Broken: Null values detected inside Title fields."
if "id" in df_converted.columns:
    assert not df_converted.duplicated(subset=["id"]).any(), "Validation Broken: Duplicate asset keys found."

print("Integrity validations successfully complete. Dataset complies with schemas.")

final_clean_path = PROJECT_ROOT / "data" / "processed" / "cleaned" / "clean.csv"
df_converted.to_csv(final_clean_path, index=False)
print(f"Persisted clean dataset to production target: {final_clean_path}")

--- Step 6: Final Structural System Validation ---
Integrity validations successfully complete. Dataset complies with schemas.
Persisted clean dataset to production target: c:\xampp\htdocs\Art-Museum-Collection-Pipeline\data\processed\cleaned\clean.csv
